In [1]:
%%writefile merge_sort_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <time.h>

#define N (1 << 20)   // 1M elements

// ---------------- CPU MERGE SORT ----------------
void merge(int arr[], int l, int m, int r) {
    int i, j, k;
    int n1 = m - l + 1;
    int n2 = r - m;

    int *L = (int*)malloc(n1 * sizeof(int));
    int *R = (int*)malloc(n2 * sizeof(int));

    for (i = 0; i < n1; i++) L[i] = arr[l + i];
    for (j = 0; j < n2; j++) R[j] = arr[m + 1 + j];

    i = 0; j = 0; k = l;
    while (i < n1 && j < n2) {
        arr[k++] = (L[i] <= R[j]) ? L[i++] : R[j++];
    }

    while (i < n1) arr[k++] = L[i++];
    while (j < n2) arr[k++] = R[j++];

    free(L);
    free(R);
}

void mergeSortCPU(int arr[], int l, int r) {
    if (l < r) {
        int m = (l + r) / 2;
        mergeSortCPU(arr, l, m);
        mergeSortCPU(arr, m + 1, r);
        merge(arr, l, m, r);
    }
}

// ---------------- GPU MERGE STEP ----------------
__device__ void merge_device(int *arr, int *temp, int left, int mid, int right) {
    int i = left;
    int j = mid;
    int k = left;

    while (i < mid && j < right) {
        if (arr[i] < arr[j]) temp[k++] = arr[i++];
        else temp[k++] = arr[j++];
    }

    while (i < mid) temp[k++] = arr[i++];
    while (j < right) temp[k++] = arr[j++];
}

// Kernel for parallel merging
__global__ void mergeKernel(int *arr, int *temp, int width, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int left = idx * 2 * width;

    if (left < n) {
        int mid = min(left + width, n);
        int right = min(left + 2 * width, n);

        merge_device(arr, temp, left, mid, right);
    }
}

// ---------------- GPU MERGE SORT ----------------
void mergeSortGPU(int *h_arr, int n) {
    int *d_arr, *d_temp;

    cudaMalloc((void**)&d_arr, n * sizeof(int));
    cudaMalloc((void**)&d_temp, n * sizeof(int));

    cudaMemcpy(d_arr, h_arr, n * sizeof(int), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    for (int width = 1; width < n; width *= 2) {
        mergeKernel<<<blocks, threads>>>(d_arr, d_temp, width, n);
        cudaDeviceSynchronize();

        // Swap arrays
        int *temp = d_arr;
        d_arr = d_temp;
        d_temp = temp;
    }

    cudaMemcpy(h_arr, d_arr, n * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_arr);
    cudaFree(d_temp);
}

// ---------------- MAIN ----------------
int main() {
    int *arr_cpu = (int*)malloc(N * sizeof(int));
    int *arr_gpu = (int*)malloc(N * sizeof(int));

    // Initialize random data
    for (int i = 0; i < N; i++) {
        arr_cpu[i] = rand() % 10000;
        arr_gpu[i] = arr_cpu[i];
    }

    // -------- CPU TIMING --------
    clock_t start_cpu = clock();
    mergeSortCPU(arr_cpu, 0, N - 1);
    clock_t end_cpu = clock();
    double cpu_time = ((double)(end_cpu - start_cpu)) / CLOCKS_PER_SEC;

    // -------- GPU TIMING --------
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    mergeSortGPU(arr_gpu, N);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    // -------- OUTPUT --------
    printf("CPU Time: %f seconds\n", cpu_time);
    printf("GPU Time: %f ms\n", gpu_time);

    // Verify correctness
    for (int i = 0; i < N; i++) {
        if (arr_cpu[i] != arr_gpu[i]) {
            printf("Mismatch at index %d\n", i);
            return -1;
        }
    }

    printf("Sorting verified successfully!\n");

    free(arr_cpu);
    free(arr_gpu);

    return 0;
}

Writing merge_sort_cuda.cu


In [2]:
!nvcc merge_sort_cuda.cu -o merge_sort
!./merge_sort

CPU Time: 0.387120 seconds
GPU Time: 0.000000 ms
Mismatch at index 0


In [ ]:
%%writefile parallel_search.cu
#include <stdio.h>
#include <cuda.h>

#define N (1 << 20)

// GPU Kernel
__global__ void parallelSearch(int *arr, int key, int *result) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        if (arr[idx] == key) {
            *result = idx;
        }
    }
}       

// CPU Search
int cpuSearch(int *arr, int key) {
    for (int i = 0; i < N; i++) {
        if (arr[i] == key) return i;
    }
    return -1;
}

int main() {
    int *arr = (int*)malloc(N * sizeof(int));
    int *d_arr, *d_result;
    int result = -1;

    // Initialize array
    for (int i = 0; i < N; i++) arr[i] = i;

    int key = N - 10;

    cudaMalloc(&d_arr, N * sizeof(int));
    cudaMalloc(&d_result, sizeof(int));

    cudaMemcpy(d_arr, arr, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &result, sizeof(int), cudaMemcpyHostToDevice);

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    parallelSearch<<<(N+255)/256, 256>>>(d_arr, key, d_result);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(&result, d_result, sizeof(int), cudaMemcpyDeviceToHost);

    // CPU timing
    clock_t c1 = clock();
    int cpu_result = cpuSearch(arr, key);
    clock_t c2 = clock();

    printf("GPU Found at: %d, Time: %f ms\n", result, gpu_time);
    printf("CPU Found at: %d, Time: %f sec\n", cpu_result,
           (double)(c2-c1)/CLOCKS_PER_SEC);

    cudaFree(d_arr);
    cudaFree(d_result);
    free(arr);

    return 0;
}

Writing parallel_search.cu


In [4]:
!nvcc parallel_search.cu -o search
!./search

GPU Found at: 1048566, Time: 26.523680 ms
CPU Found at: 1048566, Time: 0.003007 sec
